##Fase 6 · Gobierno y consumo 

#### Dar permisos de lectura

In [0]:
%sql
-- Conceder accesos en cascada para consumo analítico
GRANT USE CATALOG ON CATALOG dev_casopractico TO `account users`;
GRANT USE SCHEMA ON SCHEMA dev_casopractico.gold TO `account users`;
GRANT SELECT ON SCHEMA dev_casopractico.gold TO `account users`;


#### Creamos máscara para columna PII

In [0]:
%sql
-- 1. Crear la función de enmascaramiento
CREATE OR REPLACE FUNCTION dev_casopractico.gold.email_mask(email STRING)
RETURN CASE 
  WHEN is_account_group_member('pii_readers') THEN email 
  ELSE 'REDACTED_EMAIL_PII' 
END;

-- 2. Aplicar la máscara a la columna email de tu dimensión Gold manual
ALTER TABLE dev_casopractico.gold.dim_clientes_manual 
ALTER COLUMN email SET MASK dev_casopractico.gold.email_mask;


#### Crear filtro para ventas

In [0]:
%sql
-- 1. Crear la función que actúa como filtro de fila
CREATE OR REPLACE FUNCTION dev_casopractico.gold.tienda_filter(tienda_id STRING)
RETURN is_account_group_member('sales_all') OR tienda_id = 'TIENDA_MADRID_01';

-- 2. Aplicar el filtro de fila a tu tabla de ventas de Lakeflow
ALTER TABLE dev_casopractico.silver.ventas_silver_lf 
SET ROW FILTER dev_casopractico.gold.tienda_filter ON (tienda_id);


#### Enriquecimiento Semántico y Optimización

In [0]:
%sql
-- Añadir comentarios explicativos a nivel de tabla y columna
COMMENT ON TABLE dev_casopractico.gold.dim_clientes_manual IS 'Dimensión maestra de clientes limpios de la cadena con registro histórico SCD Tipo 2';
COMMENT ON COLUMN dev_casopractico.gold.dim_clientes_manual.email IS 'Dirección de correo electrónico protegida bajo políticas de privacidad de datos personales PII';

-- Aplicar Liquid Clustering para optimizar las consultas y búsquedas analíticas por cliente
ALTER TABLE dev_casopractico.gold.dim_clientes_manual CLUSTER BY (cliente_id);


#### Verificación Criterio Aceptación

In [0]:
%sql
SELECT cliente_id, nombre, email FROM dev_casopractico.gold.dim_clientes_manual LIMIT 5;




In [0]:
%sql
SELECT DISTINCT tienda_id FROM dev_casopractico.silver.ventas_silver_lf;